In [1]:
from pathlib import Path
import os
import sys

import pandas as pd

cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise RuntimeError("Could not infer project root from notebook location.")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

print(f"Project root: {PROJECT_ROOT}")


Project root: /Users/ourmangg/Documents/Personal_Project/LLMAgora


In [2]:
from dataclasses import fields
import gc as _gc
import json as _json
import os as _os
import string as _string

# Let any op that isn't yet implemented in Metal fall back to CPU instead of erroring.
_os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import torch as _torch

from agora.eval_aggregate import build_experiment_analysis_record
from agora.experiment import ExperimentConfig, SEMANTIC_ANALYSIS_METRICS, SEMANTIC_SIMILARITY_METHOD_COSINE
from agora.semantic_similarity_analyzer import DEFAULT_COSINE_MODEL_NAME, DEFAULT_NLI_MODEL_NAME
from agora.emotion_analyzer import DEFAULT_EMOTION_MODEL
from agora.sweep_results import SweepManifest

import agora.debate_history as _dh
import agora.emotion_analyzer as _ea_mod
import agora.eval_aggregate as _eag_mod
import agora.semantic_similarity_analyzer as _ssa_mod


def _select_device() -> str:
    """Pick the fastest local accelerator: mps on Apple Silicon, cuda on NVIDIA, else cpu."""
    if _torch.backends.mps.is_available() and _torch.backends.mps.is_built():
        return "mps"
    if _torch.cuda.is_available():
        return "cuda"
    return "cpu"


DEVICE = _select_device()
print(f"Selected device: {DEVICE}")

EXPERIMENT_CONFIG_FIELDS = {field.name for field in fields(ExperimentConfig)}

# ---- Choose which sweep to analyze ----
SWEEP_NUMBER = 6

SWEEP_RESULTS_ROOT = PROJECT_ROOT / "Sweep_results"
SWEEP_DIR = SWEEP_RESULTS_ROOT / f"sweep_{SWEEP_NUMBER}"
MANIFEST_PATH = SWEEP_DIR / "manifest.json"
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Manifest not found for sweep {SWEEP_NUMBER}: {MANIFEST_PATH}"
    )

# Persona adherence and any other LLM-API-call analyses are intentionally OFF.
postpro = {
    "device": DEVICE,
    "enable_semantic_analysis": True,
    "enable_nli": True,
    "nli_model_name": DEFAULT_NLI_MODEL_NAME,
    "enable_emotions": True,
    "emotion_model_name": DEFAULT_EMOTION_MODEL,
    "semantic_similarity_method": SEMANTIC_SIMILARITY_METHOD_COSINE,
    "semantic_similarity_model": DEFAULT_COSINE_MODEL_NAME,
    "persona_analysis_metrics": [],
}

postpro["semantic_analysis_metrics"] = (
    list(SEMANTIC_ANALYSIS_METRICS) if postpro["enable_semantic_analysis"] else []
)
postpro["semantic_similarity_device"] = postpro["device"]

analysis_kwargs = {
    key: value for key, value in postpro.items() if key in EXPERIMENT_CONFIG_FIELDS
}
analysis_kwargs["catalog_path"] = str(PROJECT_ROOT / "data" / "scenarios.json")
analysis_kwargs["prompts_path"] = str(PROJECT_ROOT / "data" / "prompts.json")

# ---- Stance-stripping infrastructure ----
# Strip the leading scenario decision label (e.g. "PROMOTE", "DO NOT ENDORSE",
# "SUBMIT NOW") from each public utterance and private reflection before any
# semantic / NLI / emotion analyzer reads them. Decision-classification metrics
# in agora.sweep_results read agora.structured_history() directly and are
# unaffected by this patch (they still see the raw stance prefix).

_SCENARIOS_PATH = PROJECT_ROOT / "data" / "scenarios.json"
SCENARIO_DECISION_LABELS = {
    s["scenario_id"]: list(s.get("decision_labels", []))
    for s in _json.loads(_SCENARIOS_PATH.read_text(encoding="utf-8")).get("scenarios", [])
}

_STANCE_CONTEXT = {"decision_labels": []}
_STRIP_LEADING_CHARS = "".join(set(_string.whitespace + ":,.;-\u2013\u2014!?\"'`*\u2026"))


def _strip_stance_prefix(text, decision_labels):
    if not text or not decision_labels:
        return text
    leading_ws = len(text) - len(text.lstrip())
    body = text[leading_ws:]
    upper = body.upper()
    for label in sorted(decision_labels, key=lambda x: -len(x)):
        if not label:
            continue
        if upper.startswith(label.upper()):
            return body[len(label):].lstrip(_STRIP_LEADING_CHARS)
    return text


_original_get_structured_debate_history = _dh.get_structured_debate_history


def _patched_get_structured_debate_history(structured_history):
    result = _original_get_structured_debate_history(structured_history)
    labels = _STANCE_CONTEXT.get("decision_labels") or []
    if not labels:
        return result
    for _slot, agent_data in result.items():
        for turn in agent_data.get("debate_turns", []):
            turn["public_speech"] = _strip_stance_prefix(turn.get("public_speech", ""), labels)
            turn["private_reflection"] = _strip_stance_prefix(turn.get("private_reflection", ""), labels)
    return result


_dh.get_structured_debate_history = _patched_get_structured_debate_history
_ssa_mod.get_structured_debate_history = _patched_get_structured_debate_history
_ea_mod.get_structured_debate_history = _patched_get_structured_debate_history


# ---- Accelerator memory cleanup ----
# _serialize_nli_all_repeats and _serialize_emotions_all_repeats call
# _serialize_nli / _serialize_emotions once per repeat, each of which
# instantiates a fresh CrossEncoder / HF pipeline. Without an explicit
# empty_cache between calls the MPS allocator (and to a lesser extent CUDA)
# accumulates inactive model memory across many groups, eventually OOM-ing
# even though only one model is live at a time. We wrap the per-repeat
# serializers to release accelerator memory after each call.

def _release_accelerator_memory():
    _gc.collect()
    if DEVICE == "mps" and hasattr(_torch, "mps"):
        try:
            _torch.mps.empty_cache()
        except (RuntimeError, AttributeError):
            pass
    elif DEVICE == "cuda" and _torch.cuda.is_available():
        _torch.cuda.empty_cache()


_orig_serialize_nli = _eag_mod._serialize_nli
_orig_serialize_emotions = _eag_mod._serialize_emotions


def _serialize_nli_with_cleanup(group_result, *, model_name, device):
    try:
        return _orig_serialize_nli(group_result, model_name=model_name, device=device)
    finally:
        _release_accelerator_memory()


def _serialize_emotions_with_cleanup(group_result, *, model_name, device):
    try:
        return _orig_serialize_emotions(group_result, model_name=model_name, device=device)
    finally:
        _release_accelerator_memory()


_eag_mod._serialize_nli = _serialize_nli_with_cleanup
_eag_mod._serialize_emotions = _serialize_emotions_with_cleanup

print("Scenario decision labels:")
for sid, labels in SCENARIO_DECISION_LABELS.items():
    print(f"  {sid}: {labels}")

analysis_kwargs


Selected device: mps
Scenario decision labels:
  ngo_climate_endorsement: ['ENDORSE', 'DO NOT ENDORSE']
  promotion_committee: ['PROMOTE', 'DO NOT PROMOTE']
  faculty_manuscript_submission: ['SUBMIT NOW', 'DELAY SUBMISSION']


{'semantic_similarity_method': 'cosine',
 'semantic_similarity_model': 'all-mpnet-base-v2',
 'persona_analysis_metrics': [],
 'semantic_analysis_metrics': ['self_consistency',
  'cross_agent_public_alignment',
  'cross_agent_private_alignment'],
 'semantic_similarity_device': 'mps',
 'catalog_path': '/Users/ourmangg/Documents/Personal_Project/LLMAgora/data/scenarios.json',
 'prompts_path': '/Users/ourmangg/Documents/Personal_Project/LLMAgora/data/prompts.json'}

In [3]:
import time

start_time = time.time()

manifest = SweepManifest.from_path(MANIFEST_PATH)

records = []
for index, group in enumerate(manifest):
    scenario_id = group.sweep_values.get("scenario_id")
    _STANCE_CONTEXT["decision_labels"] = SCENARIO_DECISION_LABELS.get(scenario_id, [])
    try:
        record = build_experiment_analysis_record(
            group,
            manifest.sweep_root,
            experiment_index=index,
            analysis_kwargs=analysis_kwargs,
            include_nli=postpro["enable_nli"],
            nli_model_name=postpro["nli_model_name"],
            include_emotions=postpro["enable_emotions"],
            emotion_model_name=postpro["emotion_model_name"],
            device=postpro["device"],
        )
    finally:
        _STANCE_CONTEXT["decision_labels"] = []
        _release_accelerator_memory()
    if record:
        records.append(record)

aggregate_df_stance_drop = pd.DataFrame(records)

end_time = time.time()

print(f"Rows: {len(aggregate_df_stance_drop)}")
display(
    aggregate_df_stance_drop[
        [
            "experiment_index",
            "model",
            "incentive_type",
            "incentive_direction",
            "scenario_id",
            "repeat_count",
        ]
    ].head()
)

elapsed_time = end_time - start_time
hours = int(elapsed_time // 3600)
minutes = int((elapsed_time % 3600) // 60)
seconds = int(elapsed_time % 60)
print(f"Time taken: {hours} hr {minutes} min {seconds} sec")

/Users/ourmangg/Documents/Personal_Project/LLMAgora/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading cosine model: all-mpnet-base-v2...
Loading cosine model: all-mpnet-base-v2...
Loading cosine model: all-mpnet-base-v2...
Loading cosine model: all-mpnet-base-v2...
Loading cosine model: all-mpnet-base-v2...
Loading nli model: dleemiller/finecat-nli-l...
Loading nli model: dleemiller/finecat-nli-l...
Loading nli model: dleemiller/finecat-nli-l...
Loading nli model: dleemiller/finecat-nli-l...
Loading nli model: dleemiller/finecat-nli-l...
Loading nli model: dleemiller/finecat-nli-l...
Loading emotion model: cirimus/modernbert-base-emotions...


Device set to use mps
Compiling the model with `torch.compile` and using a `torch.mps` device is not supported. Falling back to non-compiled mode.


Loading emotion model: cirimus/modernbert-base-emotions...


Device set to use mps


...

Device set to use mps


Loading emotion model: cirimus/modernbert-base-emotions...


Device set to use mps


Loading cosine model: all-mpnet-base-v2...
Loading cosine model: all-mpnet-base-v2...
Loading cosine model: all-mpnet-base-v2...
Loading cosine model: all-mpnet-base-v2...
Loading cosine model: all-mpnet-base-v2...
Loading nli model: dleemiller/finecat-nli-l...
Loading nli model: dleemiller/finecat-nli-l...
Loading nli model: dleemiller/finecat-nli-l...
Loading nli model: dleemiller/finecat-nli-l...


RuntimeError: MPS backend out of memory (MPS allocated: 17.96 GiB, other allocations: 8.33 MiB, max allowed: 18.13 GiB). Tried to allocate 196.75 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [ ]:
import pickle

stance_drop_pickle_path = SWEEP_DIR / f"aggregate_df_stance_drop_sweep_{SWEEP_NUMBER}.pkl"
with open(stance_drop_pickle_path, "wb") as f:
    pickle.dump(aggregate_df_stance_drop, f)
print(f"Saved: {stance_drop_pickle_path}")

In [ ]:
aggregate_df_stance_drop

,experiment_index,config_fingerprint,repeat_count,case_ids,model,incentive_direction,incentive_type,scenario_id,cosine-similarity-self-consistency,cosine-similarity-cross-agent-alignment,cosine-similarity-self-consistency-all-repeats,cosine-similarity-cross-agent-alignment-all-repeats,persona-individual-turn-scores,persona-cumulative-scores,persona-full-debate-scores,persona-individual-turn-scores-all-repeats,persona-cumulative-scores-all-repeats,persona-full-debate-scores-all-repeats,survey-public,survey-private,survey-diff-public-minus-private,survey-public-all-repeats,survey-private-all-repeats,survey-diff-public-minus-private-all-repeats,decision-self-consistency,decision-cross-agent-alignment,decision-self-consistency-all-repeats,decision-cross-agent-alignment-all-repeats,nli-self-consistency,nli-cross-agent-alignment,nli-self-consistency-all-repeats,nli-cross-agent-alignment-all-repeats,emotion-public-utterances,emotion-private-reflections,emotion-public-utterances-all-repeats,emotion-private-reflections-all-repeats
0,0,d6a2325b3ba91aba3075ea727443263ae9db9ed5813053...,5,"[91d76aeeb231, ce8ab417fef9, 483640c34bcc, af8...",openai/gpt-5.4,None,None,promotion_committee,"{'alpha': {'debate_turn': [1, 2, 3, 4, 5], 'co...","{'public alignment': {'debate_turn': [1, 2, 3,...","{'repeats': [{'alpha': {'debate_turn': [1, 2, ...",{'repeats': [{'public alignment': {'debate_tur...,"{'alpha': {'public': {'debate_turn': [1, 2, 3,...","{'alpha': {'public': {'debate_turn': [], 'pers...","{'alpha': {'public': {'score': None, 'standard...",{'repeats': [{'alpha': {'public': {'debate_tur...,{'repeats': [{'alpha': {'public': {'debate_tur...,{'repeats': [{'alpha': {'public': {'score': No...,{'Q1': {'question': 'I agree with the other pa...,{'Q1': {'question': 'I agree with the other pa...,{'Q1': {'question': 'I agree with the other pa...,{'repeats': [{'Q1': {'question': 'I agree with...,{'repeats': [{'Q1': {'question': 'I agree with...,{'repeats': [{'Q1': {'question': 'I agree with...,"{'decision': 'PROMOTE', 'channel_tuple_orderin...","{'decision': 'PROMOTE', 'agent_tuple_ordering'...","{'decision': 'PROMOTE', 'channel_tuple_orderin...","{'decision': 'PROMOTE', 'agent_tuple_ordering'...","{'alpha': {'debate_turns': [1, 2, 3, 4, 5], 'n...","{'public utterances': {'debate_turns': [1, 2, ...","{'repeats': [{'alpha': {'debate_turns': [1, 2,...",{'repeats': [{'public utterances': {'debate_tu...,"{'alpha': {'debate_turns': [1, 2, 3, 4, 5], 'e...","{'alpha': {'debate_turns': [1, 2, 3, 4, 5], 'e...","{'repeats': [{'alpha': {'debate_turns': [1, 2,...","{'repeats': [{'alpha': {'debate_turns': [1, 2,..."
1,1,241aaa7a50b69462b45cec64b5689b9f32d192e8525ac7...,5,"[4fc4a2fd3f1b, 136b484765e0, 71153033bce3, f40...",openai/gpt-5.4,None,None,ngo_climate_endorsement,"{'alpha': {'debate_turn': [1, 2, 3, 4, 5], 'co...","{'public alignment': {'debate_turn': [1, 2, 3,...","{'repeats': [{'alpha': {'debate_turn': [1, 2, ...",{'repeats': [{'public alignment': {'debate_tur...,"{'alpha': {'public': {'debate_turn': [1, 2, 3,...","{'alpha': {'public': {'debate_turn': [], 'pers...","{'alpha': {'public': {'score': None, 'standard...",{'repeats': [{'alpha': {'public': {'debate_tur...,{'repeats': [{'alpha': {'public': {'debate_tur...,{'repeats': [{'alpha': {'public': {'score': No...,{'Q1': {'question': 'I agree with the other pa...,{'Q1': {'question': 'I agree with the other pa...,{'Q1': {'question': 'I agree with the other pa...,{'repeats': [{'Q1': {'question': 'I agree with...,{'repeats': [{'Q1': {'question': 'I agree with...,{'repeats': [{'Q1': {'question': 'I agree with...,"{'decision': 'ENDORSE', 'channel_tuple_orderin...","{'decision': 'ENDORSE', 'agent_tuple_ordering'...","{'decision': 'ENDORSE', 'channel_tuple_orderin...","{'decision': 'ENDORSE', 'agent_tuple_ordering'...","{'alpha': {'debate_turns': [1, 2, 3, 4, 5], 'n...","{'public utterances': {'debate_turns': [1, 2, ...","{'repeats': [{'alpha': {'debate_turns': [1, 2,...",{'repeats': [{'public utterances': {'debat

In [ ]:
experiment_index = 0
row = aggregate_df_stance_drop.loc[
    aggregate_df_stance_drop["experiment_index"] == experiment_index
].iloc[0]

print("Available aggregate columns:")
for col in aggregate_df_stance_drop.columns:
    print(f"- {col}")

row["cosine-similarity-self-consistency"]


Available aggregate columns:
- experiment_index
- config_fingerprint
- repeat_count
- case_ids
- model
- incentive_direction
- incentive_type
- scenario_id
- cosine-similarity-self-consistency
- cosine-similarity-cross-agent-alignment
- cosine-similarity-self-consistency-all-repeats
- cosine-similarity-cross-agent-alignment-all-repeats
- persona-individual-turn-scores
- persona-cumulative-scores
- persona-full-debate-scores
- persona-individual-turn-scores-all-repeats
- persona-cumulative-scores-all-repeats
- persona-full-debate-scores-all-repeats
- survey-public
- survey-private
- survey-diff-public-minus-private
- survey-public-all-repeats
- survey-private-all-repeats
- survey-diff-public-minus-private-all-repeats
- decision-self-consistency
- decision-cross-agent-alignment
- decision-self-consistency-all-repeats
- decision-cross-agent-alignment-all-repeats
- nli-self-consistency
- nli-cross-agent-alignment
- nli-self-consistency-all-repeats
- nli-cross-agent-alignment-all-repeats
- 

{'alpha': {'debate_turn': [1, 2, 3, 4, 5],
  'cosine_similarity': [0.8703784823417664,
   0.7597646236419677,
   0.8673394918441772,
   0.8469323515892029,
   0.9070197939872742],
  'standard_error': [0.019508696709571196,
   0.03722323083406751,
   0.026344051870895134,
   0.03083279370239125,
   0.00483340407534653]},
 'beta': {'debate_turn': [1, 2, 3, 4, 5],
  'cosine_similarity': [0.9056435227394104,
   0.8868520021438598,
   0.9024076342582703,
   0.9151450157165527,
   0.9201358556747437],
  'standard_error': [0.013063545397120513,
   0.007692503171212734,
   0.0047186568308502244,
   0.00967698710669307,
   0.010772808188931237]}}

In [ ]:
row["nli-self-consistency"] if "nli-self-consistency" in aggregate_df_stance_drop.columns else None


{'alpha': {'debate_turns': [1, 2, 3, 4, 5],
  'nli_probabilities': [(0.6890787770273163,
    0.31047762408852575,
    0.0004435895185451955),
   (0.5478889578953385, 0.45156189827248455, 0.0005491311938385479),
   (0.5862751342821866, 0.4132688008248806, 0.000456112579558976),
   (0.6422056811745278, 0.35735597452148793, 0.0004383489664178342),
   (0.7536603756248951, 0.2458858343306929, 0.00045380836963886394)],
  'nli_probabilities_standard_error': [(0.08992363525248509,
    0.08994311715892622,
    8.671767823994158e-05),
   (0.13405124805902027, 0.13399120361762265, 0.00010682058527325046),
   (0.11039143362823566, 0.11033489670604996, 8.828316643911406e-05),
   (0.12549271866901934, 0.12539375370396644, 0.0001366221125454084),
   (0.06287318146875161, 0.06289599656085784, 2.4570815337648186e-05)],
  'nli_tuple_ordering': ('entailment', 'neutral', 'contradiction')},
 'beta': {'debate_turns': [1, 2, 3, 4, 5],
  'nli_probabilities': [(0.7844003662467003,
    0.2144594612531364,
    0

In [ ]:
row["survey-public"]


{'Q1': {'question': 'I agree with the other participant’s overall position on this decision.',
  'question_group': 'deliberative',
  'alpha': {'debate_turn': [1, 2, 3, 4, 5],
   'response_score': [-1.0, -2.0, -2.0, -2.0, -2.0],
   'standard_error': [0.0, 0.0, 0.0, 0.0, 0.0]},
  'beta': {'debate_turn': [1, 2, 3, 4, 5],
   'response_score': [-2.0, -2.0, -2.0, -2.0, -2.0],
   'standard_error': [0.0, 0.0, 0.0, 0.0, 0.0]}},
 'Q10': {'question': 'Publicly opposing this promotion could harm working relationships with senior leadership.',
  'question_group': 'incentive',
  'alpha': {'debate_turn': [1, 2, 3, 4, 5],
   'response_score': [1.0, 1.0, 1.0, 1.0, 1.0],
   'standard_error': [0.0, 0.0, 0.0, 0.0, 0.0]},
  'beta': {'debate_turn': [1, 2, 3, 4, 5],
   'response_score': [0.8, 0.2, 0.6, 0.0, 0.0],
   'standard_error': [0.17888543819998318,
    0.1788854381999832,
    0.21908902300206645,
    0.0,
    0.0]}},
 'Q11': {'question': 'Expressing disagreement with influential committee members can 

In [ ]:
row['nli-cross-agent-alignment-all-repeats']['repeats'][0]

{'public utterances': {'debate_turns': [1, 2, 3, 4, 5],
  'nli_probabilities': [(0.001955597985215718,
    0.03287668386474252,
    0.9651676714420319),
   (0.0002259599459648598, 0.0015618834004271775, 0.9982120990753174),
   (0.0006075515848351642, 0.002233725070254877, 0.9971587061882019),
   (0.002425990125630051, 0.023908375296741724, 0.9736656844615936),
   (0.0009643136872909963, 0.007211669813841581, 0.9918240010738373)],
  'nli_probabilities_standard_error': [(0.0, 0.0, 0.0),
   (0.0, 0.0, 0.0),
   (0.0, 0.0, 0.0),
   (0.0, 0.0, 0.0),
   (0.0, 0.0, 0.0)],
  'nli_tuple_ordering': ('entailment', 'neutral', 'contradiction')},
 'private reflections': {'debate_turns': [1, 2, 3, 4, 5],
  'nli_probabilities': [(0.0002716736671573017,
    0.0018420007545500994,
    0.9978863298892975),
   (0.00012946192509843968, 0.0009215021127602085, 0.9989490509033203),
   (0.0009500115957052913, 0.013143445030436851, 0.9859065413475037),
   (0.0001579930139996577, 0.0013174850319046527, 0.99852457